In [184]:
import os
import boto3
import pandas as pd
from trino.dbapi import connect
import logging
from io import BytesIO
import tempfile
logging.basicConfig(level=logging.INFO)

endpoint_url = os.environ["MINIO_ENDPOINT"] = "http://localhost:9000"
access_key = os.environ["MINIO_ROOT_USER"] = "minio"
secret_key = os.environ["MINIO_ROOT_PASSWORD"] = "minio1234"
trino_host = os.environ["TRINO_HOST"] = "localhost"
trino_port = os.environ["TRINO_PORT"] = "8090"
trino_user = os.environ["TRINO_USER"] = "root"

In [ ]:
BRONZE_SCHEMA_LOCATION = "s3://bck-bronze/warehouse/prueba"
BRONZE_TABLE_LOCATION = "s3://bck-bronze/master/"
schema_location = BRONZE_SCHEMA_LOCATION
table_location = BRONZE_TABLE_LOCATION

In [ ]:
#variables

src_bucket = "bck-landing"
src_path = "data/data_prueba_tecnica.csv"
tgt_bucket = "bck-bronze"
tgt_path = "master/data_prueba_tecnica.parquet"


In [172]:
#prueba de lectura de datos desde minio
try:
    s3 = boto3.client(
        "s3",
        endpoint_url=endpoint_url,
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        region_name=os.getenv("AWS_REGION", "us-east-1"),
    )
    csv_bytes = s3.get_object(Bucket=src_bucket, Key=src_path)["Body"].read()
    raw = pd.read_csv(
        BytesIO(csv_bytes),
        dtype="string",
        keep_default_na=True
    )
except Exception as e:
    logging.error(f"Error connecting to MinIO: {e}")


In [ ]:
raw["amount"] = pd.to_numeric(raw["amount"],errors="coerce")
raw["created_at"] = pd.to_datetime(raw["created_at"], errors="coerce").dt.date
raw["paid_at"] = pd.to_datetime(raw["paid_at"], errors="coerce").dt.date

In [159]:
#Analisi de la información
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          9997 non-null   string 
 1   name        9997 non-null   string 
 2   company_id  9996 non-null   string 
 3   amount      10000 non-null  Float64
 4   status      10000 non-null  string 
 5   created_at  9997 non-null   object 
 6   paid_at     6009 non-null   object 
dtypes: Float64(1), object(2), string(4)
memory usage: 556.8+ KB


In [108]:
raw.columns

Index(['id', 'name', 'company_id', 'amount', 'status', 'created_at',
       'paid_at'],
      dtype='object')

In [154]:
#Analisi de la información
raw.describe()

,amount
count,10000.0
mean,2999999999999999778178897805312.0
std,299999999999999998084088103698432.0
min,0.0
25%,31.21
50%,60.605
75%,109.57
max,29999999999999997214335425004437504.0


In [110]:
raw.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,9997,9997,4e85c4eac968c9465fc8d34bcd4968ac019fa850,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
name,9997,4,MiPasajefy,9899,NaN,NaN,NaN,NaN,NaN,NaN,NaN
company_id,9996,3,cbf1c8b09cd5b549416d49d220a40cbd317f952e,9899,NaN,NaN,NaN,NaN,NaN,NaN,NaN
amount,10000.0,<NA>,<NA>,<NA>,2999999999999999778178897805312.0,299999999999999998084088103698432.0,0.0,31.21,60.605,109.57,29999999999999997214335425004437504.0
status,10000,10,paid,5892,NaN,NaN,NaN,NaN,NaN,NaN,NaN
created_at,9997,140,2019-03-01,117,NaN,NaN,NaN,NaN,NaN,NaN,NaN
paid_at,6009,140,2019-02-27,71,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [111]:
raw[raw['id'].isna()]

,id,name,company_id,amount,status,created_at,paid_at
272,<NA>,MiPasajefy,cbf1c8b09cd5b549416d49d220a40cbd317f952e,66.16,pending_payment,2019-03-14,NaT
9915,<NA>,MiPasajefy,cbf1c8b09cd5b549416d49d220a40cbd317f952e,55.71,paid,2019-02-13,2019-02-13
9917,<NA>,MiPasajefy,cbf1c8b09cd5b549416d49d220a40cbd317f952e,89.36,paid,2019-04-17,2019-04-17


In [112]:
#notamos que id realmente es único
#Name, company_id, y status son pequeños podemos analizar sus valores unicos
print(raw['company_id'].unique().tolist())
print(raw['name'].unique().tolist())
print(raw['status'].unique().tolist())
#con los prints vemos valores anómalos en las tres columnas, por lo que se hace necesario limpiarlos

['cbf1c8b09cd5b549416d49d220a40cbd317f952e', '8f642dc67fccf861548dfe1c761ce22f795e91f0', <NA>, '*******']
['MiPasajefy', 'Muebles chidos', <NA>, 'MiPas0xFFFF', 'MiP0xFFFF']
['voided', 'pending_payment', 'paid', 'pre_authorized', 'refunded', 'charged_back', 'expired', 'p&0x3fid', '0xFFFF', 'partially_refunded']


In [113]:
raw[raw['name'].isna()]

,id,name,company_id,amount,status,created_at,paid_at
731,ec79a21ef969c7fc6beef080ff56baf0aeeca8b5,<NA>,cbf1c8b09cd5b549416d49d220a40cbd317f952e,112.52,paid,2019-02-14,2019-02-14
2199,6a6ac16d53a02ba7948bff0a534e45404e716c5b,<NA>,cbf1c8b09cd5b549416d49d220a40cbd317f952e,244.88,voided,2019-05-09,NaT
2200,4740cf2624c3b929d9944cdfcb5c87e71a82ddc5,<NA>,cbf1c8b09cd5b549416d49d220a40cbd317f952e,118.78,paid,2019-03-05,2019-03-06


In [114]:
raw[raw['company_id'].isna()]

,id,name,company_id,amount,status,created_at,paid_at
262,6b77d36f2ebd5e53a76195c6c1678a5027022f9d,MiPasajefy,<NA>,3.0,pending_payment,2019-02-01,NaT
2378,28445567bf15d6751367e3828f39c255546cc1e1,MiPasajefy,<NA>,3.0,pending_payment,2019-03-23,NaT
2445,654695699dc08392248aedef372f64f0284ecb68,MiPasajefy,<NA>,30.8,paid,2019-03-27,2019-03-27
5981,6f6e718b9993ac97ff6e19c61498fb8be82320df,MiPasajefy,<NA>,69.55,voided,2019-05-10,NaT


In [115]:
raw[['name','company_id']].drop_duplicates()[['name','company_id']]

,name,company_id
0,MiPasajefy,cbf1c8b09cd5b549416d49d220a40cbd317f952e
78,Muebles chidos,8f642dc67fccf861548dfe1c761ce22f795e91f0
262,MiPasajefy,<NA>
603,MiPasajefy,*******
731,<NA>,cbf1c8b09cd5b549416d49d220a40cbd317f952e
1320,MiPas0xFFFF,cbf1c8b09cd5b549416d49d220a40cbd317f952e
1479,MiP0xFFFF,cbf1c8b09cd5b549416d49d220a40cbd317f952e


In [116]:
cat_cols = raw.select_dtypes(include="string").columns
for col in cat_cols:
    print(f"\n### {col}")
    print(raw[col].value_counts(dropna=False).head(10))


### id
id
<NA>                                        3
59268c70f1e2a4372e06b7e8c308d556141a81d0    1
0e066d3752a543428d7dc81c537adaad7a4a4bca    1
4637367cf394b2e314e82fde7cde695491cd4449    1
b7243030b5ecb367fed8f42b5961e6cf39532a63    1
221b7af73b7901a32c526553e40a289724dd023a    1
4e85c4eac968c9465fc8d34bcd4968ac019fa850    1
48ba4bdbfb56ceebb32f2bd0263e759be942af3d    1
05fc6f5ac66b6ee7e4253aa5d0c2299eb47aaaf4    1
2cdce231c1fc6a2061bfa2f1d978351fe217245d    1
Name: count, dtype: Int64

### name
name
MiPasajefy        9899
Muebles chidos      96
<NA>                 3
MiPas0xFFFF          1
MiP0xFFFF            1
Name: count, dtype: Int64

### company_id
company_id
cbf1c8b09cd5b549416d49d220a40cbd317f952e    9899
8f642dc67fccf861548dfe1c761ce22f795e91f0      96
<NA>                                           4
*******                                        1
Name: count, dtype: Int64

### status
status
paid                  5892
voided                2084
pending_payment       188

In [117]:
raw[~raw["status"].isin(["paid", "voided", "pending_payment", "pre_authorized", "refunded", "charged_back", "expired", "partially_refunded"])]

,id,name,company_id,amount,status,created_at,paid_at
1308,9c99ef0dc472daaf059be45c8dd24a3781fd1af7,MiPasajefy,cbf1c8b09cd5b549416d49d220a40cbd317f952e,82.36,p&0x3fid,2019-03-14,2019-03-14
3510,8ee1ad3b1736f801145e18ecf54e346c006e43d5,MiPasajefy,cbf1c8b09cd5b549416d49d220a40cbd317f952e,146.74,0xFFFF,2019-02-03,2019-02-03


In [118]:
raw[~raw["company_id"].str.fullmatch(r"[0-9a-f]{40}", na=False)]

,id,name,company_id,amount,status,created_at,paid_at
262,6b77d36f2ebd5e53a76195c6c1678a5027022f9d,MiPasajefy,<NA>,3.0,pending_payment,2019-02-01,NaT
603,701ed7d3e728c41ecf58639f64de12d7dc2fc4df,MiPasajefy,*******,92.66,paid,2019-02-27,2019-02-27
2378,28445567bf15d6751367e3828f39c255546cc1e1,MiPasajefy,<NA>,3.0,pending_payment,2019-03-23,NaT
2445,654695699dc08392248aedef372f64f0284ecb68,MiPasajefy,<NA>,30.8,paid,2019-03-27,2019-03-27
5981,6f6e718b9993ac97ff6e19c61498fb8be82320df,MiPasajefy,<NA>,69.55,voided,2019-05-10,NaT


In [119]:
raw.groupby("company_id")["name"].nunique().sort_values(ascending=False).head(20)

company_id
cbf1c8b09cd5b549416d49d220a40cbd317f952e    3
*******                                     1
8f642dc67fccf861548dfe1c761ce22f795e91f0    1
Name: name, dtype: int64

In [120]:
raw[raw['company_id']=='cbf1c8b09cd5b549416d49d220a40cbd317f952e']['name'].unique()

<StringArray>
['MiPasajefy', <NA>, 'MiPas0xFFFF', 'MiP0xFFFF']
Length: 4, dtype: string

In [121]:
raw.groupby("name")["company_id"].nunique().sort_values(ascending=False).head(20)

name
MiPasajefy        2
MiP0xFFFF         1
MiPas0xFFFF       1
Muebles chidos    1
Name: company_id, dtype: int64

In [122]:
raw[raw['name']=='MiPasajefy']['company_id'].unique()

<StringArray>
['cbf1c8b09cd5b549416d49d220a40cbd317f952e', <NA>, '*******']
Length: 3, dtype: string

In [123]:
raw[raw['paid_at'].isna()]['status'].value_counts(dropna=False)

status
voided             2084
pending_payment    1881
pre_authorized       18
expired               8
Name: count, dtype: Int64

In [156]:
raw[raw['id']=='94f33d3d5a142c7dfcbf247806ad68cf1dc93515']

,id,name,company_id,amount,status,created_at,paid_at
1752,94f33d3d5a142c7dfcbf247806ad68cf1dc93515,MiPasajefy,cbf1c8b09cd5b549416d49d220a40cbd317f952e,0.0,voided,2019-02-11,NaT


inconsistencias
1. ID nulo en tres registros
2. company_id nulo en 4 registros
3. name vacio en algunos registros
4. status con valores anomalos (0xFFFF,MiPas0xFFFF)
5. Name con valores sospechosos (MiP0xFFFF,MiPas0xFFFF)
6. amount muy grande

In [185]:
#limpieza
stg = raw.copy() 
#1. omitir id nulos
stg = stg[stg['id'].notna()]

#2 y 3 mapear company_id a nombre y nombre a company id
name_to_company = (
    raw.dropna(subset=["company_id", "name"])
    .groupby("name", as_index=False)["company_id"]
    .first()
    .rename(columns={"company_id": "canonical_company_id"})
)

company_to_name = (
    raw.dropna(subset=["company_id", "name"])
    .groupby("company_id", as_index=False)["name"]
    .first()
    .rename(columns={"name": "canonical_name"})
)

stg = stg.merge(name_to_company, on ="name", how="left")
stg = stg.merge(company_to_name, on="company_id", how="left")
stg["company_id_filled"] = stg["company_id"].fillna(stg["canonical_company_id"])
stg["name_filled"] = stg["name"].fillna(stg["canonical_name"])
stg = stg.drop(columns=[ "canonical_company_id", "canonical_name"])

#4 columna de status
valid_statuses = {
    "paid",
    "voided",
    "pending_payment",
    "pre_authorized",
    "refunded",
    "charged_back",
    "expired",
    "partially_refunded",
}


stg["status_clean"] = stg["status"].where(stg["status"].isin(valid_statuses), "unknown")

#5. eliminar nombres sospechosos
stg.loc[stg["name"].isin(["MiP0xFFFF", "MiPas0xFFFF"]), "name_filled"] = "MiPasajefy"

#drop outliers en amount
amount_num = pd.to_numeric(stg["amount"], errors="coerce")
p99 = amount_num.quantile(0.99)

mask_amounts = (
    amount_num.notna()
    & np.isfinite(amount_num)
    & amount_num.ge(0)
    & amount_num.le(p99)
)

stg_rejected = stg[~mask_amounts].copy()
stg = stg[mask_amounts]


#result 
stg = stg.drop(columns=["company_id", "name", "status"])
stg = stg.rename(columns={"company_id_filled": "company_id", "name_filled": "name", "status_clean": "status"})

stg["is_paid"] = stg["status"].eq("paid")
stg["pendig_payment"] = stg["status"].eq("pending_payment")
stg["paid_amount"] = stg["amount"].where(stg["is_paid"], 0.0)
stg["pending_amount"] = stg["amount"].where(stg["pendig_payment"], 0.0)


#agregaciones 

aggregated = (
    stg.groupby(["name", "created_at"], as_index=False)
    .agg(
        transactions=("id", "size"),
        total_amount=("amount", "sum"),
        average_amount=("amount", "mean"),
        paid_transactions=("is_paid", "sum"),
        paid_amount=("paid_amount", "sum"),
        pending_payment_transactions=("pendig_payment", "sum"),
        pending_amount=("pending_amount", "sum"),
        min_amount=("amount", "min"),
        max_amount=("amount", "max"),
    )
    .sort_values(["created_at", "name"])
)

with tempfile.TemporaryDirectory() as tmpdir:
    parquet_path = Path(tmpdir) / "data_prueba_tecnica.parquet"
    aggregated.to_parquet(parquet_path, index=False, engine="pyarrow", compression="snappy")
    s3.upload_file(str(parquet_path), tgt_bucket, tgt_path)


print(stg.shape)
print(stg_rejected.shape)
stg.head(5)
aggregated.head(5)


(9897, 11)
(100, 10)


,name,created_at,transactions,total_amount,average_amount,paid_transactions,paid_amount,pending_payment_transactions,pending_amount,min_amount,max_amount
0,MiPasajefy,2019-01-01,59,4150.04,70.339661,18,2659.99,15,102.76,3.0,803.8
1,MiPasajefy,2019-01-02,81,10638.95,131.345062,42,3946.05,22,5940.47,3.0,2511.71
2,MiPasajefy,2019-01-03,62,6735.66,108.639677,37,4284.42,15,1846.12,3.0,1578.54
140,Muebles chidos,2019-01-03,1,3199.0,3199.0,1,3199.0,0,0.0,3199.0,3199.0
3,MiPasajefy,2019-01-04,76,6349.69,83.548553,54,5438.95,10,209.71,3.0,386.78


In [ ]:


conn = connect(
    host=os.getenv("TRINO_HOST", "trino"),
    port=int(os.getenv("TRINO_PORT", "8080")),
    user=os.getenv("TRINO_USER", "root"),
    catalog="bronze",
    schema="prueba",
    http_scheme="http",
)

cursor = conn.cursor()
try:
    cursor.execute(
        f"""
        CREATE SCHEMA IF NOT EXISTS bronze.prueba
        WITH (location = '{schema_location}')
        """
    )
    cursor.execute(
        f"""
        CREATE TABLE IF NOT EXISTS bronze.prueba.tbl_data (
            name varchar,
            created_at date,
            transaction_count bigint,
            unique_company_ids bigint,
            total_amount double,
            average_amount double,
            paid_transactions bigint,
            paid_amount double,
            min_amount double,
            max_amount double
        )
        WITH (
            external_location = '{table_location}',
            format = 'PARQUET'
        )
        """
    )
finally:
    cursor.close()
    conn.close()

return {
    "source_path": SOURCE_PATH,
    "bronze_path": BRONZE_PARQUET_PATH,
    "rows_aggregated": int(len(aggregated)),
}